<a href="https://colab.research.google.com/github/elmir-mamedov/MLOps_prj/blob/main/SCD_duplicates2json.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
path = kagglehub.dataset_download("arunrk7/surface-crack-detection")

Using Colab cache for faster access to the 'surface-crack-detection' dataset.


In [2]:
import os

base = '/kaggle/input/surface-crack-detection'

def scan_dir(p, limit=5):
    with os.scandir(p) as ents:
        count = 0
        for e in ents:
            if e.is_dir():
                print(f"Directory: {e.path}")
                scan_dir(e.path, limit)
            else:
                if count >= limit:
                  print(f"  ... (truncated)")
                  break
                print(f"    File: {e.path}")
            count += 1

scan_dir(base)

Directory: /kaggle/input/surface-crack-detection/Negative
    File: /kaggle/input/surface-crack-detection/Negative/08450.jpg
    File: /kaggle/input/surface-crack-detection/Negative/19812.jpg
    File: /kaggle/input/surface-crack-detection/Negative/16916.jpg
    File: /kaggle/input/surface-crack-detection/Negative/05938.jpg
    File: /kaggle/input/surface-crack-detection/Negative/06122.jpg
  ... (truncated)
Directory: /kaggle/input/surface-crack-detection/Positive
    File: /kaggle/input/surface-crack-detection/Positive/08450.jpg
    File: /kaggle/input/surface-crack-detection/Positive/19812.jpg
    File: /kaggle/input/surface-crack-detection/Positive/05938.jpg
    File: /kaggle/input/surface-crack-detection/Positive/06122.jpg
    File: /kaggle/input/surface-crack-detection/Positive/08536.jpg
  ... (truncated)


In [3]:
import subprocess
result = subprocess.run(['find', base, '-name', '*.jpg'], capture_output=True, text=True)

from collections import Counter
folders = Counter(os.path.dirname(p) for p in result.stdout.strip().split('\n'))
for folder, count in sorted(folders.items()):
    print(f"{folder}: {count} images")

/kaggle/input/surface-crack-detection/Negative: 20000 images
/kaggle/input/surface-crack-detection/Positive: 20000 images


In [4]:
import subprocess
from collections import Counter

result = subprocess.run(
    f"find {base} -name '*.jpg' -printf '%s %p\n'",
    capture_output=True, text=True, shell=True
)

from collections import Counter
sizes = [line.split()[0] for line in result.stdout.strip().split('\n')]
dupes = {s: c for s, c in Counter(sizes).items() if c > 1}
print(f"Total: {len(sizes)}")
print(f"Same-size groups: {len(dupes)}")

Total: 40000
Same-size groups: 5013


In [5]:
import subprocess
from collections import Counter

result = subprocess.run(
    f"find {base} -name '*.jpg' -print0 | xargs -0 md5sum",
    capture_output=True, text=True, shell=True
)

hashes = [line.split()[0] for line in result.stdout.strip().split('\n')]
dupes = {h: c for h, c in Counter(hashes).items() if c > 1}
print(f"Total images: {len(hashes)}")
print(f"Duplicate groups: {len(dupes)}")
print(f"Duplicate files: {sum(dupes.values()) - len(dupes)}")

Total images: 40000
Duplicate groups: 1519
Duplicate files: 1598


In [6]:
lines = result.stdout.strip().split('\n')
hash_to_files = {}
for line in lines:
    h, path = line.split(None, 1)
    hash_to_files.setdefault(h, []).append(path)

# Check duplicates
same_class = 0
cross_class = 0
for h, files in hash_to_files.items():
    if len(files) > 1:
        folders = set(os.path.dirname(f) for f in files)
        if len(folders) == 1:
            same_class += 1
        else:
            cross_class += 1

print(f"Duplicates within same class: {same_class}")
print(f"Duplicates across classes: {cross_class}")

Duplicates within same class: 1519
Duplicates across classes: 0


In [7]:
import json

# Build duplicate info
duplicates = {}
for h, files in hash_to_files.items():
    if len(files) > 1:
        duplicates[h] = {
            "keep": files[0],
            "remove": files[1:]
        }

# Save
output = {
    "total_images": 40000,
    "duplicate_groups": len(duplicates),
    "files_to_remove": sum(len(d["remove"]) for d in duplicates.values()),
    "duplicates": duplicates
}

with open("duplicates.json", "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved {len(duplicates)} groups to duplicates.json")

Saved 1519 groups to duplicates.json
